# seq

In [ ]:
///- --test static

open testing

## seq

### seq

In [ ]:
type seq dim el = dim -> option el

### try_item

In [ ]:
inl try_item n s =
    n |> s

### from_list

In [ ]:
inl from_list list =
    fun n =>
        list
        |> listm'.try_item n

In [1]:
///- --test

listm.init 10i32 print_and_return
|> from_list
|> try_item 5i32
|> _assert_eq (Some 5i32)

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
print_and_return{ x = 6 }
print_and_return{ x = 7 }
print_and_return{ x = 8 }
print_and_return{ x = 9 }
__assert_eq { expected = Some(5) }

### map

In [ ]:
inl map fn s =
    fun n =>
        n
        |> s
        |> optionm.map fn

In [2]:
///- --test

listm.init 10i32 id
|> from_list
|> map ((*) 2)
|> try_item 5i32
|> _assert_eq (Some 10i32)

__assert_eq { expected = Some(10) }

### mapi

In [ ]:
inl mapi fn s =
    fun n =>
        n
        |> s
        |> optionm.map (fn n)

In [3]:
///- --test

listm.init 10i32 print_and_return
|> from_list
|> mapi fun i x => i + x
|> try_item 5i32
|> _assert_eq (Some 10i32)

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
print_and_return{ x = 6 }
print_and_return{ x = 7 }
print_and_return{ x = 8 }
print_and_return{ x = 9 }
__assert_eq { expected = Some(10) }

### choose

In [ ]:
inl choose forall dim {number} t u. (fn : t -> option u) (s : seq dim t) : seq dim u =
    fun n =>
        inl rec body fn s i i' =
            match i |> s with
            | None => None
            | Some x =>
                match x |> fn with
                | Some x when n = i' => Some x
                | Some _ => 루프 (i + 1) (i' + 1)
                | _ => 루프 (i + 1) i'
        and inl 루프 i i' =
            if n |> var_is |> not
            then body fn s i i'
            else
                inl fn = join fn
                inl s = join s
                join body fn s i i'
        루프 0 0

In [4]:
///- --test

listm.init 10i32 print_and_return
|> from_list
|> choose (fun x => if x % 2 = 0 then Some x else None)
|> try_item 1i32
|> _assert_eq (Some 2i32)

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
print_and_return{ x = 6 }
print_and_return{ x = 7 }
print_and_return{ x = 8 }
print_and_return{ x = 9 }
__assert_eq { expected = Some(2) }

### indexed

In [ ]:
inl indexed s =
    s
    |> mapi fun i x =>
        i, x

In [5]:
///- --test

listm.init 10i32 ((*) 2)
|> from_list
|> indexed
|> try_item 5i32
|> _assert_eq (Some (5i32, 10i32))

__assert_eq { expected = Some(5, 10) }

### zip

In [ ]:
inl zip n seq1 seq2 =
    seq1 n, seq2 n

In [6]:
///- --test

((listm.init 10i32 id |> from_list), (listm.init 10i32 ((*) 2) |> from_list))
||> zip 5i32
|> _assert_eq (Some 5, Some 10)

__assert_eq { expected = (Some(5), Some(10)) }

### zip_with

In [ ]:
inl zip_with fn seq1 seq2 =
    fun n =>
        fn (seq1 n) (seq2 n)

In [7]:
///- --test

((listm.init 10i32 id |> from_list), (listm.init 10i32 ((*) 2) |> from_list))
||> zip_with (optionm'.choose (+))
|> try_item 2i32
|> _assert_eq (Some 6)

__assert_eq { expected = Some(6) }

### fold

In [ ]:
inl fold fn init seq =
    inl rec 루프 acc n =
        match seq n with
        | Some x => 루프 (fn acc x) (n + 1)
        | None => acc
    루프 init 0

inl fold_ fn init seq =
    let rec 루프 acc n =
        match seq n with
        | Some x => 루프 (fn acc x) (n + 1)
        | None => acc
    루프 init 0

### sum

In [ ]:
inl sum seq =
    seq |> fold (+) 0

inl sum_ seq =
    seq |> fold_ (+) 0

In [8]:
///- --test

listm.init 10i32 id
|> from_list
|> fun f (n : i32) => f n
|> sum
|> _assert_eq 45

__assert_eq { expected = 45 }

### to_list

In [ ]:
inl to_list seq =
    seq
    |> fold (fun acc x => x :: acc) []
    |> listm.rev

inl to_list_ seq =
    seq
    |> fold_ (fun acc x => x :: acc) []
    |> listm.rev

In [9]:
///- --test

listm.init 10i32 id
|> from_list
|> fun f (n : i32) => f n
|> to_list
|> _assert_eq (listm.init 10i32 id)

__assert_eq { expected = Cons(0, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(7, Cons(8, Cons(9, Nil)))))))))) }

### from_array

In [ ]:
inl from_array forall dim {number; int} el. (array : a dim el) : seq dim el =
    fun n =>
        if n >= length array
        then None
        else index array n |> Some

In [10]:
///- --test

a ;[ 1; 2; 3 ]
|> from_array
|> try_item 1i32
|> _assert_eq (Some 2i32)

__assert_eq { expected = Some(2) }

### to_array

In [ ]:
inl to_array seq =
    inl ar = a ;[] |> mut
    ((), seq)
    ||> fold fun _ x =>
        ar <- *ar ++ a ;[x]
    *ar

inl to_array_ seq =
    inl ar = a ;[] |> mut
    ((), seq)
    ||> fold_ fun _ x =>
        ar <- *ar ++ a ;[x]
    *ar

In [11]:
///- --test

listm.init 10i32 id
|> from_list
|> fun (x : i32 -> _) => x
|> to_array
|> _assert_eq (a ;[ 0; 1; 2; 3; 4; 5; 6; 7; 8; 9 ] : _ i32 _)

__assert_eq { expected = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] }

### take_while

In [ ]:
inl take_while cond seq =
    inl rec 루프 acc i =
        match seq i with
        | Some st when cond st i => 루프 (st :: acc) (i + 1)
        | _ => acc |> listm.rev
    루프 [] 0

In [12]:
///- --test

listm.init 10i32 id
|> from_list
|> take_while (fun n (_ : i32) => n < 5)
|> listm'.sum
|> _assert_eq 10

__assert_eq { expected = 10 }

In [13]:
///- --test

stream.new_finite_stream print_and_return 10i32
|> flip stream.try_item
|> take_while (fun n (_ : i32) => n < 5)
|> listm'.sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }

### take_while_

In [ ]:
inl take_while_ cond seq =
    let rec 루프 acc i =
        match seq i with
        | Some st when cond st i => 루프 (st :: acc) (i + 1)
        | _ => acc |> listm.rev
    루프 [] 0

In [14]:
///- --test

stream.new_infinite_stream_ print_and_return
|> flip stream.try_item
|> take_while_ (fun n (_ : i32) => n < 5i32)
|> listm'.sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }

In [15]:
///- --test

stream.new_infinite_stream_ print_and_return
|> stream.memoize
|> fun list =>
    inl list = list ()
    fun n =>
        list |> stream.try_item n
|> take_while_ (fun n (_ : i32) => n < 5i32)
|> listm'.sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }

In [16]:
///- --test

stream.new_finite_stream print_and_return 10i32
|> stream.memoize
|> fun list =>
    inl list = list ()
    fun n =>
        list |> stream.try_item n
|> take_while_ (fun n (_ : i32) => n < 5)
|> listm'.sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }

### memoize

In [ ]:
inl memoize seq =
    inl state = mut []
    fun n =>
        match *state |> listm'.try_find (fun (n', _) => n' = n) with
        | Some (_, v) => v
        | None =>
            inl new_state = seq n
            state <- (n, new_state) :: *state
            new_state

inl memoize_ seq =
    inl state = mut []
    fun n =>
        match *state |> listm'.try_find_ (fun (n', _) => n' = n) with
        | Some (_, v) => v
        | None =>
            inl new_state = seq n
            state <- (n, new_state) :: *state
            new_state

In [17]:
///- --test

inl seq =
    fun n =>
        n |> print_and_return |> Some
    |> memoize_

seq
|> take_while_ (fun n (_ : i32) => n < 5)
|> listm'.sum
|> _assert_eq 10

seq
|> take_while_ (fun n _ => n < 5)
|> listm'.sum
|> _assert_eq 10

print_and_return{ x = 0 }
print_and_return{ x = 1 }
print_and_return{ x = 2 }
print_and_return{ x = 3 }
print_and_return{ x = 4 }
print_and_return{ x = 5 }
__assert_eq { expected = 10 }
__assert_eq { expected = 10 }

### iterate

In [ ]:
inl iterate forall a b {number}. (f : a -> a) (x0 : a) (num_steps : b) : a =
    inl rec 루프 forall c {number}. (x : a) (n : c) : a =
        if n <= 0
        then x
        else 루프 (f x) (n - 1)
    루프 x0 num_steps

In [18]:
///- --test

10i32 |> iterate ((*) 2) 1i32
|> _assert_eq 1024

__assert_eq { expected = 1024 }

In [ ]:
inl iterate_ forall a b {number}. (f : a -> a) (x0 : a) (num_steps : b) : a =
    let rec 루프 forall c {number}. (x : a) (n : c) : a =
        if n <= 0
        then x
        else 루프 (f x) (n - 1)
    루프 x0 num_steps

In [19]:
///- --test

10i32 |> iterate_ ((*) 2) 1i32
|> _assert_eq 1024

__assert_eq { expected = 1024 }

In [ ]:
inl iterate' f x0 num_steps =
    listm.init num_steps id
    |> listm.fold (fun x _ => f x) x0

In [20]:
///- --test

10i32 |> iterate' ((*) 2) 1i32
|> _assert_eq 1024

__assert_eq { expected = 1024 }

### find_last

In [ ]:
inl find_last forall item result. fold_fn fn target : option result =
    fold_fn (fun (item : item) (result : option result) =>
        match result with
        | None => fn item
        | result => result
    ) target (None : option result)

inl array_find_last forall item result. (fn : item -> option result) (target : a i32 item) : option result =
    find_last am.foldBack fn target

inl list_find_last forall item result. (fn : item -> option result) (target : list item) : option result =
    find_last listm.foldBack fn target

## fsharp

### seq'

In [ ]:
type seq_switch' t =
    {
        Gleam : $'List(`t)'
        Fsharp : $'`t seq'
        Python : $'list'
        Rust : $'Rc<Vec<`t>>'
        TypeScript : $'Array<`t>'
        Zig : $'[]const `(t)'
    }
nominal seq' t = $'backend_switch `(seq_switch' t)'

### length'

In [ ]:
inl length' forall t. (items : seq' t) : int =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/list"
            $'!items |> list.length' : int
        Fsharp = fun () => items |> $'Seq.length' : int
        Python = fun () => $'len(!items)' : int
        Rust = fun () => $'(!items.len() as `int)' : int
        TypeScript = fun () => $'!items.length' : int
        Zig = fun () => $'\@as(`int, \@intCast(!items.len))' : int
    }

### to_list'

In [ ]:
inl to_list' forall t. (items : seq' t) : listm'.list' t =
    backend_switch {
        Gleam = fun () => $'!items ' : listm'.list' t
        Fsharp = fun () => items |> $'Seq.toList' : listm'.list' t
        Python = fun () => $'!items ' : listm'.list' t
        Rust = fun () => $'(!items).clone()' : listm'.list' t
        TypeScript = fun () => $'[...!items]' : listm'.list' t
        Zig = fun () => $'!items ' : listm'.list' t
    }

### new_seq

In [ ]:
inl new_seq forall t. fn : seq' t =
    backend_switch {
        Gleam = fun () =>
            $'!fn()' : seq' t
        Fsharp = fun () =>
            fun () =>
                $'seq {'
                fn |> indent
                $'}' : ()
            |> let'
            |> fun x => x : seq' t
        Python = fun () =>
            $'list(!fn())' : seq' t
    }

In [21]:
///- --test
///> fsharp
///> cuda

fun () =>
    "a" |> yield
    "b" |> yield
|> new_seq
|> to_list'
|> listm'.unbox
|> _assert_eq [ "a"; "b" ]

.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }

### of_array'

In [ ]:
inl of_array' forall dim t. (items : a dim t) : seq' t =
    backend_switch {
        Gleam = fun () => $'!items |> spiral_array_to_list' : seq' t
        Fsharp = fun () =>
            fun () =>
                $'for i = 0 to !items.Length - 1 do yield !items.[i]'
            |> new_seq
            |> fun x => x : seq' t
        Python = fun () => $'[item for item in !items]' : seq' t
        Rust = fun () => $'Rc::new(!items.borrow().clone())' : seq' t
        TypeScript = fun () => $'[...!items]' : seq' t
        Zig = fun () => $'!items ' : seq' t
    }

In [22]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

(a ;[ "a"; "b" ] : _ int _)
|> of_array'
|> to_list'
|> listm'.unbox
|> _assert_eq [ "a"; "b" ]

.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }

0

### of_array

In [ ]:
inl of_array forall dim t. (items : a dim t) : seq' t =
    backend_switch {
        Gleam = fun () => $'!items |> spiral_array_to_list' : seq' t
        Fsharp = fun () => $'!items |> Seq.ofArray' : seq' t
        Python = fun () => $'list(iter(!items))' : seq' t
        Rust = fun () => $'Rc::new(!items.borrow().clone())' : seq' t
        TypeScript = fun () => $'[...!items]' : seq' t
        Zig = fun () => $'!items ' : seq' t
    }

In [23]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

(a ;[ "a"; "b" ] : _ int _)
|> of_array
|> to_list'
|> listm'.unbox
|> _assert_eq [ "a"; "b" ]

.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }

0

### of_array_base

In [ ]:
inl of_array_base forall t. (items : array_base t) : seq' t =
    a items
    |> fun x => x : _ int _
    |> of_array

In [24]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

;[ "a"; "b" ]
|> of_array_base
|> to_list'
|> listm'.unbox
|> _assert_eq [ "a"; "b" ]

.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }

0

### to_array'

In [ ]:
inl to_array' forall dim t. (items : seq' t) : a dim t =
    backend_switch {
        Gleam = fun () => $'!items |> spiral_array_from_list' : a dim t
        Fsharp = fun () => items |> $'Seq.toArray' : a dim t
        Python = fun () =>
            inl x = ($'!items ' : listm'.list' t) |> listm'.to_array'
            $'!x ' : a dim t
        Rust = fun () => $'Rc::new(RefCell::new((*!items).clone()))' : a dim t
        TypeScript = fun () => $'[...!items]' : a dim t
        Zig = fun () => $'(spiral_gpa.dupe(`t, !items) catch \@panic("out of memory"))' : a dim t
    }

In [25]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

;[ "a"; "b" ]
|> of_array_base
|> to_array'
|> fun x => x : _ int _
|> am'.to_list'
|> listm'.unbox
|> _assert_eq [ "a"; "b" ]

.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(a, Cons(b, Nil)) }

0

### of_list'

In [ ]:
inl of_list' forall t. (items : listm'.list' t) : seq' t =
    backend_switch {
        Gleam = fun () => $'!items ' : seq' t
        Fsharp = fun () =>
            fun () =>
                items |> yield_from
            |> new_seq
            |> fun x => x : seq' t
        Python = fun () => $'!items ' : seq' t
        Rust = fun () => $'(!items).clone()' : seq' t
        TypeScript = fun () => $'[...!items]' : seq' t
        Zig = fun () => $'!items ' : seq' t
    }

### cast'

In [ ]:
inl cast' forall t u. (items : u) : seq' t =
    backend_switch {
        Gleam = fun () => $'!items ' : seq' t
        Fsharp = fun () => items |> $'Seq.cast' : seq' t
        Python = fun () => $'list(!items)' : seq' t
        Rust = fun () =>
            inl x : seq' t =
                real
                    typecase u with
                    | seq' ~x => $'(!items).clone()' : seq' t
                    | listm'.list' ~x => $'(!items).clone()' : seq' t
                    | array_base ~x => $'Rc::new(!items.borrow().clone())' : seq' t
                    | a ~d ~x => $'Rc::new(!items.borrow().clone())' : seq' t
                    | _ => $'Rc::new(IntoIterator::into_iter(!items).collect::<Vec<`t>>())' : seq' t
            x
        TypeScript = fun () => $'Array.from(!items)' : seq' t
        Zig = fun () => $'!items ' : seq' t
    }

### rev'

In [ ]:
inl rev'' forall t. (items : seq' t) : seq' t =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/list"
            $'!items |> list.reverse' : seq' t
        Fsharp = fun () => items |> $'Seq.rev' : seq' t
        Python = fun () => $'list(reversed(!items))' : seq' t
        Rust = fun () => $'Rc::new({ let mut v = (!items).as_ref().clone(); v.reverse(); v })' : seq' t
        TypeScript = fun () => $'[...!items].reverse()' : seq' t
        Zig = fun () => $'blk: { const r = spiral_gpa.dupe(`t, !items) catch \@panic("out of memory"); std.mem.reverse(`t, r); break :blk r; }' : seq' t
    }

In [26]:
///- --test
///> fsharp
///> cuda
///> rust
///> typescript

[ "a"; "b" ]
|> listm'.box
|> of_list'
|> rev''
|> to_list'
|> listm'.unbox
|> _assert_eq [ "b"; "a" ]

.fsx (fsharp)
__assert_eq { expected = Cons(b, Cons(a, Nil)) }
.py (cuda)
__assert_eq { expected = Cons(b, Cons(a, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(b, Cons(a, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(b, Cons(a, Nil)) }

0

## rust

### fuse

In [27]:
nominal fuse t =
    `(
        rust_type
            `($'core_iter_Fuse<`t>')
            `($'core::iter::Fuse<`t>')
            "type core_iter_Fuse<'T> = class end"
    )